# Compare four-step pipeline lemmas with spaCy-only lemmas

This notebook compares lemma assignments at each stage of the four-step SCOPE pipeline for:

1. all 105,992 SCOPE words; and
2. the 13,850 words retained for factor analysis.

The original Excel files are read with `pandas.read_excel()` so that missing values and Excel errors are interpreted exactly as in the production pipeline. The saved spaCy-only output is used as the comparison condition; spaCy is not rerun here.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

REPO_DIR = Path('/work/desai-lab/xuanyang/Project/Semantic/dissemination/github/DiscoFMRI')
SCOPE_DIR = REPO_DIR / 'data' / 'SCOPE'
OUTPUT_DIR = SCOPE_DIR / 'lemma_comparison'
OUTPUT_DIR.mkdir(exist_ok=True)

SPACY_FILE = OUTPUT_DIR / 'SCOPE_lemmatization_output_spacyonly.csv'
FA_FILE = SCOPE_DIR / 'SCOPE_lemma_var106.csv'
UK_FILE = SCOPE_DIR / 'SUBTLEX-UK.xlsx'
US_FILE = SCOPE_DIR / 'SUBTLEX-US frequency list with PoS and Zipf information.xlsx'


/home/xy6/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Load the saved spaCy results and SUBTLEX lookup tables

Reading the `Word` and lemma columns with `keep_default_na=False` preserves legitimate entries such as `nan` and `null`. The Excel lookup tables retain the production pipeline's default missing-value behavior.


In [2]:
df_spacy = pd.read_csv(
    SPACY_FILE,
    usecols=['Word', 'Lemma_spacy', 'PoS_tag_spacy'],
    keep_default_na=False,
).drop_duplicates('Word', keep='first')

df_uk = pd.read_excel(UK_FILE, usecols=['Spelling', 'DomPoSLemma'])
df_us = pd.read_excel(US_FILE, usecols=['Word', 'Dom_PoS_SUBTLEX'])

# Match the set_index(...).join(...) behavior of the original pipeline.
assert not df_uk['Spelling'].dropna().duplicated().any(), 'Duplicate SUBTLEX-UK spelling keys'
assert not df_us['Word'].dropna().duplicated().any(), 'Duplicate SUBTLEX-US word keys'

df_compare = (
    df_spacy
    .merge(df_uk, left_on='Word', right_on='Spelling', how='left')
    .merge(df_us, on='Word', how='left')
)

print(f'{len(df_compare):,} unique SCOPE words loaded.')
display(df_compare.head())


105,992 unique SCOPE words loaded.


,Word,Lemma_spacy,PoS_tag_spacy,Spelling,DomPoSLemma,Dom_PoS_SUBTLEX
0,'em,them,NOUN,NaN,NaN,NaN
1,'neath,'neath,PUNCT,NaN,NaN,NaN
2,'re,be,AUX,NaN,NaN,NaN
3,'shun,'shun,PROPN,NaN,NaN,NaN
4,'tis,'tis,VERB,NaN,NaN,NaN


## Reconstruct Steps 1–3

- **Step 1:** use `DomPoSLemma` from SUBTLEX-UK.
- **Step 2:** where Step 1 is unavailable, use SUBTLEX-US dominant POS to guide NLTK's WordNet lemmatizer. POS values outside adjective, adverb, verb, and noun default to noun, matching the original function.
- **Step 3:** where neither preceding source is available, use the saved spaCy lemma.


In [3]:
pos_map = {
    'Adjective': wordnet.ADJ,
    'Adverb': wordnet.ADV,
    'Verb': wordnet.VERB,
    'Noun': wordnet.NOUN,
}
lemmatizer = WordNetLemmatizer()

has_uk = df_compare['DomPoSLemma'].notna()
has_us = df_compare['Dom_PoS_SUBTLEX'].notna()

df_compare['Pipeline_step'] = np.select(
    [has_uk, ~has_uk & has_us],
    [1, 2],
    default=3,
).astype(int)
df_compare['Pipeline_source'] = df_compare['Pipeline_step'].map({
    1: 'Step 1: SUBTLEX-UK',
    2: 'Step 2: SUBTLEX-US + NLTK',
    3: 'Step 3: spaCy fallback',
})

df_compare['Lemma_before_step4'] = df_compare['Lemma_spacy']
df_compare.loc[has_uk, 'Lemma_before_step4'] = df_compare.loc[has_uk, 'DomPoSLemma']

step2_mask = ~has_uk & has_us
df_compare.loc[step2_mask, 'Lemma_before_step4'] = [
    lemmatizer.lemmatize(word, pos_map.get(pos, wordnet.NOUN))
    for word, pos in zip(
        df_compare.loc[step2_mask, 'Word'],
        df_compare.loc[step2_mask, 'Dom_PoS_SUBTLEX'],
    )
]

expected_stage_counts = {1: 61842, 2: 5717, 3: 38433}
actual_stage_counts = df_compare['Pipeline_step'].value_counts().sort_index().to_dict()
assert actual_stage_counts == expected_stage_counts, (actual_stage_counts, expected_stage_counts)
print('Stage counts:', actual_stage_counts)


Stage counts: {1: 61842, 2: 5717, 3: 38433}


## Reconstruct Step 4

Step 4 retains the raw word form as the lemma for entries containing selected punctuation and for the explicitly listed informal or contracted forms.


In [4]:
punctuation = ["'", '_', '-', '.']
explicit_special_cases = {
    'cant', 'couldnt', 'wont', 'gonna', 'wanna', 'kinda',
    'cannot', 'gotta', 'hes', 'id', 'Id', 'shes', 'thats', 'theres',
    'theyre', 'wed', 'whats', 'whos', 'whys',
}

df_compare['Step4_special_case'] = (
    df_compare['Word'].apply(lambda word: any(mark in word for mark in punctuation))
    | df_compare['Word'].isin(explicit_special_cases)
)
df_compare['Lemma_pipeline_final'] = df_compare['Lemma_before_step4']
df_compare.loc[df_compare['Step4_special_case'], 'Lemma_pipeline_final'] = (
    df_compare.loc[df_compare['Step4_special_case'], 'Word']
)

assert df_compare['Step4_special_case'].sum() == 2108
print(f"{df_compare['Step4_special_case'].sum():,} Step 4 special cases.")


2,108 Step 4 special cases.


## Comparison functions

`Different_words` is the number of word forms assigned different lemmas. The distinct-lemma columns describe how many unique pipeline lemmas, spaCy lemmas, and lemma pairs occur among those disagreements.


In [5]:
def comparison_row(data, label, pipeline_column):
    different = data[pipeline_column].ne(data['Lemma_spacy'])
    disagreements = data.loc[different]
    return {
        'Comparison': label,
        'N_words': len(data),
        'Same_words': int((~different).sum()),
        'Different_words': int(different.sum()),
        'Percent_different': 100 * different.mean(),
        'Distinct_pipeline_lemmas_in_differences': disagreements[pipeline_column].nunique(),
        'Distinct_spacy_lemmas_in_differences': disagreements['Lemma_spacy'].nunique(),
        'Distinct_lemma_pairs': disagreements[[pipeline_column, 'Lemma_spacy']].drop_duplicates().shape[0],
    }


def make_summary(data):
    rows = []
    for step in (1, 2, 3):
        subset = data.loc[data['Pipeline_step'].eq(step)]
        rows.append(comparison_row(
            subset,
            subset['Pipeline_source'].iloc[0],
            'Lemma_before_step4',
        ))
    rows.append(comparison_row(data, 'All words before Step 4', 'Lemma_before_step4'))
    rows.append(comparison_row(
        data.loc[data['Step4_special_case']],
        'Step 4 special cases only',
        'Lemma_pipeline_final',
    ))
    rows.append(comparison_row(data, 'Final after Step 4', 'Lemma_pipeline_final'))
    return pd.DataFrame(rows)


def disagreement_examples(data, pipeline_column='Lemma_pipeline_final', n=25):
    columns = [
        'Word', 'Pipeline_step', 'Pipeline_source', pipeline_column,
        'Lemma_spacy', 'PoS_tag_spacy', 'Step4_special_case',
    ]
    return data.loc[data[pipeline_column].ne(data['Lemma_spacy']), columns].head(n)


## A. All SCOPE words


In [6]:
summary_all = make_summary(df_compare)
display(summary_all.style.format({'Percent_different': '{:.3f}%'}))
display(disagreement_examples(df_compare))

summary_all.to_csv(OUTPUT_DIR / 'lemma_comparison_all_SCOPE_summary.csv', index=False)
df_compare.loc[
    df_compare['Lemma_pipeline_final'].ne(df_compare['Lemma_spacy'])
].to_csv(OUTPUT_DIR / 'lemma_comparison_all_SCOPE_disagreements.csv', index=False)


,Comparison,N_words,Same_words,Different_words,Percent_different,Distinct_pipeline_lemmas_in_differences,Distinct_spacy_lemmas_in_differences,Distinct_lemma_pairs
0,Step 1: SUBTLEX-UK,61842,57423,4419,7.146%,4315,4113,4369
1,Step 2: SUBTLEX-US + NLTK,5717,5219,498,8.711%,493,494,498
2,Step 3: spaCy fallback,38433,38433,0,0.000%,0,0,0
3,All words before Step 4,105992,101075,4917,4.639%,4802,4586,4865
4,Step 4 special cases only,2108,1137,971,46.063%,971,970,971
5,Final after Step 4,105992,100106,5886,5.553%,5771,5553,5834


,Word,Pipeline_step,Pipeline_source,Lemma_pipeline_final,Lemma_spacy,PoS_tag_spacy,Step4_special_case
0,'em,3,Step 3: spaCy fallback,'em,them,NOUN,True
2,'re,3,Step 3: spaCy fallback,'re,be,AUX,True
5,'twas,3,Step 3: spaCy fallback,'twas,'twa,VERB,True
12,'ve,3,Step 3: spaCy fallback,'ve,have,AUX,True
48,abashed,1,Step 1: SUBTLEX-UK,abashed,abash,VERB,False
64,abb_s,3,Step 3: spaCy fallback,abb_s,abb_,NOUN,True
77,abbots,1,Step 1: SUBTLEX-UK,abbots,abbot,VERB,False
81,abbreviated,1,Step 1: SUBTLEX-UK,abbreviated,abbreviate,VERB,False
107,abducted,1,Step 1: SUBTLEX-UK,abducted,abduct,VERB,False
118,abed,1,Step 1: SUBTLEX-UK,abed,abe,VERB,False


## B. Words used in factor analysis

This is conditional on the 13,850-word complete-case sample selected using the original pipeline.


In [16]:
df_fa_words = pd.read_csv(
    FA_FILE,
    usecols=['Word', 'Lemma'],
    keep_default_na=False,
).rename(columns={'Lemma': 'Lemma_saved_factor_analysis'})

df_compare_fa = df_fa_words.merge(df_compare, on='Word', how='left', validate='one_to_one')
assert len(df_compare_fa) == 13850
assert df_compare_fa['Lemma_spacy'].notna().all()
assert df_compare_fa['Lemma_saved_factor_analysis'].eq(
    df_compare_fa['Lemma_pipeline_final']
).all(), 'Reconstructed lemmas do not match the saved factor-analysis lemmas'

summary_fa = make_summary(df_compare_fa)
display(summary_fa.style.format({'Percent_different': '{:.3f}%'}))
display(disagreement_examples(df_compare_fa))

summary_fa.to_csv(OUTPUT_DIR / 'lemma_comparison_factor_analysis_summary.csv', index=False)
df_compare_fa.loc[
    df_compare_fa['Lemma_pipeline_final'].ne(df_compare_fa['Lemma_spacy'])
].to_csv(OUTPUT_DIR / 'lemma_comparison_factor_analysis_disagreements.csv', index=False)


,Comparison,N_words,Same_words,Different_words,Percent_different,Distinct_pipeline_lemmas_in_differences,Distinct_spacy_lemmas_in_differences,Distinct_lemma_pairs
0,Step 1: SUBTLEX-UK,13511,13111,400,2.961%,371,390,392
1,Step 2: SUBTLEX-US + NLTK,100,82,18,18.000%,17,18,18
2,Step 3: spaCy fallback,239,239,0,0.000%,0,0,0
3,All words before Step 4,13850,13432,418,3.018%,386,407,409
4,Step 4 special cases only,1,1,0,0.000%,0,0,0
5,Final after Step 4,13850,13432,418,3.018%,386,407,409


,Word,Pipeline_step,Pipeline_source,Lemma_pipeline_final,Lemma_spacy,PoS_tag_spacy,Step4_special_case
19,abolishes,1,Step 1: SUBTLEX-UK,abolish,abolishe,NOUN,False
106,acorns,1,Step 1: SUBTLEX-UK,acorn,acorns,PROPN,False
178,advanced,1,Step 1: SUBTLEX-UK,advanced,advance,VERB,False
218,affording,1,Step 1: SUBTLEX-UK,afford,affording,ADJ,False
222,aged,1,Step 1: SUBTLEX-UK,age,aged,ADJ,False
398,annoying,1,Step 1: SUBTLEX-UK,annoy,annoying,ADJ,False
412,antennae,1,Step 1: SUBTLEX-UK,antenna,antennae,NOUN,False
440,aped,1,Step 1: SUBTLEX-UK,ape,aped,VERB,False
506,arched,1,Step 1: SUBTLEX-UK,arched,arch,VERB,False
537,aromas,1,Step 1: SUBTLEX-UK,aroma,aromas,PROPN,False


## Step 4's incremental effect

This distinguishes disagreements already present after Steps 1–3 from disagreements created or resolved by the special-case override.


In [17]:
def step4_effect(data):
    changed = (
        data['Step4_special_case']
        & data['Lemma_before_step4'].ne(data['Lemma_pipeline_final'])
    )
    agreed_before = data['Lemma_before_step4'].eq(data['Lemma_spacy'])
    agrees_after = data['Lemma_pipeline_final'].eq(data['Lemma_spacy'])
    return pd.Series({
        'Step4_special_cases': int(data['Step4_special_case'].sum()),
        'Assignments_changed_by_step4': int(changed.sum()),
        'Disagreements_created': int((changed & agreed_before & ~agrees_after).sum()),
        'Disagreements_resolved': int((changed & ~agreed_before & agrees_after).sum()),
        'Changed_but_still_different': int((changed & ~agreed_before & ~agrees_after).sum()),
    })

step4_summary = pd.DataFrame({
    'All_SCOPE': step4_effect(df_compare),
    'Factor_analysis_words': step4_effect(df_compare_fa),
})
display(step4_summary)
step4_summary.to_csv(OUTPUT_DIR / 'lemma_comparison_step4_effect.csv')


,All_SCOPE,Factor_analysis_words
Step4_special_cases,2108,1
Assignments_changed_by_step4,971,0
Disagreements_created,970,0
Disagreements_resolved,1,0
Changed_but_still_different,0,0


## Expected headline results

With the current repository files, the principal results should be:

- all SCOPE words: 4,419 Step 1 differences, 498 Step 2 differences, and no Step 3 differences before the Step 4 override;
- factor-analysis words: 400 Step 1 differences, 18 Step 2 differences, and no Step 3 differences; and
- final factor-analysis sample: 418 of 13,850 words differ (3.02%).
